In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tsfresh
from tsfresh import extract_relevant_features, select_features
from tsfresh.feature_extraction import extract_features
from tsfresh.feature_extraction.settings import EfficientFCParameters, ComprehensiveFCParameters, MinimalFCParameters
from tsfresh.utilities.dataframe_functions import impute
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import accuracy_score
from time import perf_counter
from copy import deepcopy

In [ ]:
# Parameters
n_series_per_class = 50
series_length = 100
noise_level = 0.10

data = []
ids = []
times = []
labels = []

current_id = 0
t = np.linspace(0, 2 * np.pi, series_length)

# -------- CLASS 0: Base sine --------
for _ in range(n_series_per_class):
    y = np.sin(t)
    y += np.random.normal(0, noise_level, series_length)

    data.extend(y)
    ids.extend([current_id] * series_length)
    times.extend(range(series_length))
    labels.append(0)
    current_id += 1

# -------- CLASS 1: Base sine + weak 2nd harmonic --------
for _ in range(n_series_per_class):
    amp = np.random.uniform(0.02, 0.05)
    phase = np.random.uniform(0, 2 * np.pi)

    y = np.sin(t)
    y += amp * np.sin(2 * t + phase)
    y += np.random.normal(0, noise_level, series_length)

    data.extend(y)
    ids.extend([current_id] * series_length)
    times.extend(range(series_length))
    labels.append(1)
    current_id += 1

# -------- CLASS 2: Base sine + weak 4rd harmonic --------
for _ in range(n_series_per_class):
    amp = np.random.uniform(0.02, 0.05)
    phase = np.random.uniform(0, 2 * np.pi)

    y = np.sin(t)
    y += amp * np.sin(4 * t + phase)
    y += np.random.normal(0, noise_level, series_length)

    data.extend(y)
    ids.extend([current_id] * series_length)
    times.extend(range(series_length))
    labels.append(2)
    current_id += 1

# -------- CLASS 3: Base sine + weak 8th harmonic --------
for _ in range(n_series_per_class):
    amp = np.random.uniform(0.02, 0.05)
    phase = np.random.uniform(0, 2 * np.pi)

    y = np.sin(t)
    y += amp * np.sin(8 * t + phase)
    y += np.random.normal(0, noise_level, series_length)

    data.extend(y)
    ids.extend([current_id] * series_length)
    times.extend(range(series_length))
    labels.append(3)
    current_id += 1

# -------- CLASS 4: Base sine + weak 16th harmonic --------
for _ in range(n_series_per_class):
    amp = np.random.uniform(0.02, 0.05)
    phase = np.random.uniform(0, 2 * np.pi)

    y = np.sin(t)
    y += amp * np.sin(16 * t + phase)
    y += np.random.normal(0, noise_level, series_length)

    data.extend(y)
    ids.extend([current_id] * series_length)
    times.extend(range(series_length))
    labels.append(4)
    current_id += 1

# -------- CLASS 5: Square wave --------
for _ in range(n_series_per_class):
    phase = np.random.uniform(0, 2 * np.pi)

    y = np.sign(np.sin(t + phase))
    y += np.random.normal(0, noise_level, series_length)

    data.extend(y)
    ids.extend([current_id] * series_length)
    times.extend(range(series_length))
    labels.append(5)
    current_id += 1


# -------- CLASS 6: Triangle wave --------
for _ in range(n_series_per_class):
    phase = np.random.uniform(0, 2 * np.pi)

    y = (2 / np.pi) * np.arcsin(np.sin(t + phase))
    y += np.random.normal(0, noise_level, series_length)

    data.extend(y)
    ids.extend([current_id] * series_length)
    times.extend(range(series_length))
    labels.append(6)
    current_id += 1



# Create DataFrame (TSFresh format)
df = pd.DataFrame({
    "id": ids,
    "time": times,
    "value": data
})

# Labels
y = pd.Series(labels, name="target")

print(df.head())
print(y.head())

In [ ]:
# n_examples = 9

# # randomly sample IDs instead of taking the first ones
# example_ids = np.random.choice(df["id"].unique(), size=n_examples, replace=False)

# plt.figure(figsize=(12, 8))

# for i, series_id in enumerate(example_ids):
#     plt.subplot(3, 3, i + 1)
    
#     subset = df[df["id"] == series_id]
#     label = y.iloc[series_id]
    
#     plt.plot(subset["time"], subset["value"])
#     plt.title(f"ID {series_id} (Class {label})")

# plt.tight_layout()
# plt.show()

In [ ]:
# # settings = EfficientFCParameters()
# settings = MinimalFCParameters()
# # settings = ComprehensiveFCParameters()

# X = extract_features(df, column_id="id", column_sort="time", default_fc_parameters=settings)
# X = X.dropna(axis=1)
# X_selected = select_features(X, y)

In [ ]:
def fft_parameters(n):
    return [
            {"coeff": i, "attr": attr}
            for i in range(n)
            for attr in ["real", "imag", "abs", "angle"]
        ]

In [ ]:
settings = {
    "Minimal": MinimalFCParameters(),
    "Efficient": EfficientFCParameters(),
    "Comprehensive": ComprehensiveFCParameters()
}

results = []

for name, base_setting in settings.items():

    coeff_list = [0] if name == "Minimal" else [10, 25, 50, 75, 100]

    for n in coeff_list:

        setting = deepcopy(base_setting)

        if name != "Minimal":
            setting["fft_coefficient"] = fft_parameters(n)

        t1 = perf_counter()

        # -----------------------------
        # Feature extraction
        # -----------------------------
        X = extract_features(
            df,
            column_id="id",
            column_sort="time",
            default_fc_parameters=setting
        )
        extraction_time = perf_counter() - t1

        X = X.dropna(axis=1)

        extracted_features = X.shape[1]

        # -----------------------------
        # Train/Test Split
        # -----------------------------
        train_idx, test_idx = train_test_split(
            np.arange(len(y)),
            test_size=0.3,
            stratify=y,
            random_state=42
        )

        X_train = X.iloc[train_idx].copy()
        X_test = X.iloc[test_idx].copy()

        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        # -----------------------------
        # Feature Selection
        # -----------------------------
        t2 = perf_counter()
        X_selected = select_features(X_train, y_train)
        selection_time = perf_counter() - t2

        # If TSFresh removes everything, keep all extracted features
        if X_selected.shape[1] == 0:
            print(f"{name} FFT={n}: No features selected - using all extracted features.")
            X_selected = X_train

        selected_features = X_selected.shape[1]

        X_train = X_selected
        X_test = X_test[X_train.columns]

        # -----------------------------
        # Train model
        # -----------------------------
        clf = RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        )

        clf.fit(X_train, y_train)

        predictions = clf.predict(X_test)

        accuracy = accuracy_score(y_test, predictions) * 100

        elapsed = perf_counter() - t1

        results.append({
            "Method": name,
            "FFT coefficients": n,
            "Features extracted": extracted_features,
            "Features selected": selected_features,
            "Extraction Time": extraction_time,
            "Selection Time": selection_time,
            "Total Time (s)": round(elapsed, 2),
            "Accuracy (%)": round(accuracy, 2)
        })

        print(
            f"{name:14}"
            f" FFT={n:3d}"
            f" Extracted={extracted_features:4d}"
            f" Selected={selected_features:4d}"
            f" Accuracy={accuracy:6.2f}%"
        )


In [ ]:
results_df = pd.DataFrame(results)

print(results_df)

results_df.to_csv(
    "fft_sinewave_results.csv",
    index=False
)

In [125]:
walk = pd.read_csv("walking.csv")
run = pd.read_csv("running.csv")
walk = walk[(walk.time > 2.5) & (walk.time < 185)]
run = run[(run.time > 2.5) & (run.time < 130)]
walk = walk.reset_index(drop=True)
run = run.reset_index(drop=True)
walk["window_idx"] = "walk_" + (walk.index // 100).astype(str)
run["window_idx"] = "run_" + (run.index // 100).astype(str)

In [126]:
# -----------------------------
# Parameters to test
# -----------------------------
settings = {
    "Minimal": MinimalFCParameters(),
    "Efficient": EfficientFCParameters(),
    "Comprehensive": ComprehensiveFCParameters()
}

results = []

# -----------------------------
# Test each feature set
# -----------------------------
for name, base_setting in settings.items():

    coeff_list = [0] if name == "Minimal" else [5, 10, 25, 50, 75, 100]

    for n in coeff_list:

        setting = deepcopy(base_setting)

        if name != "Minimal":
            setting["fft_coefficient"] = fft_parameters(n)

        t1 = perf_counter()

        # -----------------------------
        # Feature extraction
        # -----------------------------
        combined = pd.concat([walk, run], ignore_index=True)
        t2 = perf_counter()
        X = extract_features(
            combined,
            column_id="window_idx",
            column_sort="time",
            default_fc_parameters=setting
        )
        extraction_time = perf_counter() - t2
        X = X.dropna(axis=1)

        extracted_features = X.shape[1]

        # -----------------------------
        # Labels
        # -----------------------------
        y = pd.Series(
            [0] * walk["window_idx"].nunique() +
            [1] * run["window_idx"].nunique(),
            index=X.index
        )

        # -----------------------------
        # Train / Test split
        # -----------------------------
        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            test_size=0.30,
            stratify=y,
            random_state=42
        )

        # -----------------------------
        # Feature selection
        # -----------------------------
        t3 = perf_counter()
        X_train_selected = select_features(X_train, y_train)
        selection_time = perf_counter() - t3
        # If nothing selected, keep all features
        if X_train_selected.shape[1] == 0:
            X_train_selected = X_train

        selected_features = X_train_selected.shape[1]

        X_test = X_test[X_train_selected.columns]

        # -----------------------------
        # Train classifier
        # -----------------------------
        clf = RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        )

        clf.fit(X_train_selected, y_train)

        predictions = clf.predict(X_test)

        accuracy = accuracy_score(
            y_test,
            predictions
        ) * 100

        elapsed = perf_counter() - t1

        # -----------------------------
        # Store results
        # -----------------------------
        results.append({
            "Method": name,
            "FFT coefficients": n,
            "Features extracted": extracted_features,
            "Features selected": selected_features,
            "Extraction Time": extraction_time,
            "Selection Time": selection_time,
            "Accuracy (%)": accuracy,
            "Time (s)": elapsed
        })

        print(
            f"{name:14}"
            f" FFT={n:3d}"
            f" Extracted={extracted_features:4d}"
            f" Selected={selected_features:4d}"
            f" Accuracy={accuracy:6.2f}%"
        )



Feature Extraction: 100%|██████████| 45/45 [00:03<00:00, 14.69it/s]


Minimal        FFT=  0 Extracted=  40 Selected=  31 Accuracy= 78.72%


Feature Extraction: 100%|██████████| 45/45 [00:17<00:00,  2.54it/s]


Efficient      FFT=  5 Extracted=1510 Selected= 657 Accuracy= 78.72%


Feature Extraction: 100%|██████████| 45/45 [00:18<00:00,  2.40it/s]


...

Comprehensive  FFT= 50 Extracted=1822 Selected= 737 Accuracy= 80.85%


Feature Extraction: 100%|██████████| 45/45 [00:20<00:00,  2.16it/s]


Comprehensive  FFT= 75 Extracted=1822 Selected= 737 Accuracy= 80.85%


Feature Extraction: 100%|██████████| 45/45 [00:21<00:00,  2.10it/s]


Comprehensive  FFT=100 Extracted=1822 Selected= 737 Accuracy= 80.85%


In [127]:
# -----------------------------
# Results table
# -----------------------------
results_df = pd.DataFrame(results)

print(results_df)

results_df.to_csv(
    "fft_running_results.csv",
    index=False
)

           Method  FFT coefficients  Features extracted  Features selected  \
0         Minimal                 0                  40                 31   
1       Efficient                 5                1510                657   
2       Efficient                10                1590                673   
3       Efficient                25                1798                723   
4       Efficient                50                1798                723   
5       Efficient                75                1798                723   
6       Efficient               100                1798                723   
7   Comprehensive                 5                1534                671   
8   Comprehensive                10                1614                687   
9   Comprehensive                25                1822                737   
10  Comprehensive                50                1822                737   
11  Comprehensive                75                1822         